In [30]:
import cv2
import numpy as np
import pywt
import random
import matplotlib.pyplot as plt

from pathlib import Path
import shutil

In [31]:
# SOURCE_ROOT = Path("cfp-dataset/Data/Images")
# OUTPUT_DIR = Path("cfp_flattened")

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# count = 0

# for group_dir in SOURCE_ROOT.iterdir():

#     # 001, 002, 003, ...
#     if not group_dir.is_dir():
#         continue

#     frontal_dir = group_dir / "frontal"

#     if not frontal_dir.is_dir():
#         continue

#     # Images inside frontal/
#     for image_path in frontal_dir.iterdir():

#         if not image_path.is_file():
#             continue

#         if image_path.suffix.lower() not in extensions:
#             continue

#         # Example:
#         # 001/frontal/abc.jpg
#         # -> 001_abc.jpg
#         destination = OUTPUT_DIR / f"{group_dir.name}_{image_path.name}"

#         shutil.copy2(image_path, destination)

#         count += 1

# print(f"Copied {count} images.")
# print(f"Output: {OUTPUT_DIR.resolve()}")

In [32]:
class DwtDctWatermark:
    def __init__(
        self,
        wavelet="haar",
        block_size=8,
        alpha=10.0,
        coeff1=(3, 4),
        coeff2=(4, 3),
    ):
        self.wavelet = wavelet
        self.block_size = block_size
        self.alpha = alpha
        self.coeff1 = coeff1
        self.coeff2 = coeff2

    @staticmethod
    def text_to_bits(text):
        data = text.encode("utf-8")

        bits = []

        for byte in data:
            for i in range(7, -1, -1):
                bits.append((byte >> i) & 1)

        return np.array(bits, dtype=np.uint8)

    def _embed_bit(self, block, bit):
        dct = cv2.dct(block.astype(np.float32))

        r1, c1 = self.coeff1
        r2, c2 = self.coeff2

        a = dct[r1, c1]
        b = dct[r2, c2]

        if bit == 1:
            if a <= b + self.alpha:
                dct[r1, c1] = b + self.alpha
        else:
            if b <= a + self.alpha:
                dct[r2, c2] = a + self.alpha

        return cv2.idct(dct)

    def embed_bits(self, image, bits):
        ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)

        y = ycrcb[:, :, 0].astype(np.float32)

        # 1-level Haar DWT
        ll, (lh, hl, hh) = pywt.dwt2(
            y,
            self.wavelet
        )

        block_size = self.block_size

        blocks_y = hl.shape[0] // block_size
        blocks_x = hl.shape[1] // block_size

        capacity = blocks_y * blocks_x

        if len(bits) > capacity:
            raise ValueError(
                f"Watermark requires {len(bits)} bits, "
                f"but this image can only hold {capacity} bits."
            )

        bit_index = 0

        for by in range(blocks_y):
            for bx in range(blocks_x):

                if bit_index >= len(bits):
                    break

                y0 = by * block_size
                x0 = bx * block_size

                block = hl[
                    y0:y0 + block_size,
                    x0:x0 + block_size
                ]

                hl[
                    y0:y0 + block_size,
                    x0:x0 + block_size
                ] = self._embed_bit(
                    block,
                    bits[bit_index]
                )

                bit_index += 1

            if bit_index >= len(bits):
                break

        # Inverse DWT
        watermarked_y = pywt.idwt2(
            (ll, (lh, hl, hh)),
            self.wavelet
        )

        watermarked_y = watermarked_y[
            :image.shape[0],
            :image.shape[1]
        ]

        ycrcb[:, :, 0] = np.clip(
            watermarked_y,
            0,
            255
        ).astype(np.uint8)

        return cv2.cvtColor(
            ycrcb,
            cv2.COLOR_YCrCb2BGR
        )

    def _extract_bit(self, block):
        dct = cv2.dct(
            block.astype(np.float32)
        )

        r1, c1 = self.coeff1
        r2, c2 = self.coeff2

        return int(
            dct[r1, c1] > dct[r2, c2]
        )


    
    def extract_bits(self, image, num_bits):
        """
        Extract a fixed number of watermark bits.
        """

        ycrcb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2YCrCb
        )

        y = ycrcb[:, :, 0].astype(np.float32)

        # DWT returns:
        # LL, (LH, HL, HH)
        ll, (lh, hl, hh) = pywt.dwt2(
            y,
            self.wavelet
        )

        block_size = self.block_size

        blocks_y = hl.shape[0] // block_size
        blocks_x = hl.shape[1] // block_size

        capacity = blocks_y * blocks_x

        if num_bits > capacity:
            raise ValueError(
                f"Requested {num_bits} bits, "
                f"capacity is {capacity}."
            )

        bits = []

        for by in range(blocks_y):
            for bx in range(blocks_x):

                if len(bits) >= num_bits:
                    break

                y0 = by * block_size
                x0 = bx * block_size

                block = hl[
                    y0:y0 + block_size,
                    x0:x0 + block_size
                ]

                bits.append(
                    self._extract_bit(block)
                )

            if len(bits) >= num_bits:
                break

        return np.array(
            bits,
            dtype=np.uint8
        )

In [ ]:
def calculate_psnr(original, watermarked):
    # Use OpenCV's optimized, low-memory C++ implementation
    if original.dtype == np.uint8 and watermarked.dtype == np.uint8:
        return cv2.PSNR(original, watermarked)
    
    # Fallback for floats: compute directly without duplicate array allocations
    diff = original.astype(np.float32) - watermarked.astype(np.float32)
    mse = np.mean(diff * diff)
    
    if mse == 0:
        return float('inf')
    
    max_val = 255.0 if original.max() > 1.0 else 1.0
    return float(20 * np.log10(max_val) - 10 * np.log10(mse))

In [34]:
IMAGE_DIR = Path("official_images")

image_paths = [
    p for p in IMAGE_DIR.iterdir()
    if p.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp",
    }
]

print(f"Found {len(image_paths)} images.")

random.seed(42)

selected_paths = random.sample(
    image_paths,
    217
)

for path in selected_paths:
    print(path.name)

Found 217 images.
171.jpg
034.jpg
012.jpg
198.jpg
077.jpg
069.jpg
063.jpg
041.jpg
197.jpg
032.jpg
181.jpg
222.jpg
147.jpg
028.jpg
159.jpg
115.jpg
014.jpg
013.jpg
029.jpg
061.jpg
065.jpg
137.jpg
162.jpg
223.jpg
151.jpg
056.jpg
192.jpg
174.jpg
187.jpg
213.jpg
114.jpg
062.jpg
121.jpg
158.jpg
078.jpg
007.jpg
046.jpg
186.jpg
210.jpg
094.jpg
191.jpg
045.jpg
206.jpg
093.jpg
216.jpg
207.jpg
104.jpg
030.jpg
098.jpg
095.jpg
203.jpg
074.jpg
017.jpg
124.jpg
145.jpg
037.jpg
103.jpg
026.jpg
149.jpg
082.jpg
099.jpg
155.jpg
055.jpg
023.jpg
172.jpg
064.jpg
081.jpg
167.jpg
205.jpg
031.jpg
178.jpg
184.jpg
123.jpg
100.jpg
047.jpg
101.jpg
097.jpg
059.jpg
075.jpg
024.jpg
049.jpg
220.jpg
150.jpg
125.jpg
154.jpg
076.jpg
194.jpg
090.jpg
020.jpg
211.jpg
112.jpg
010.jpg
110.jpg
188.jpg
057.jpg
040.jpg
209.jpg
033.jpg
152.jpg
079.jpg
119.jpg
176.jpg
131.jpg
126.jpg
204.jpg
070.jpg
200.jpg
089.jpg
135.jpg
170.jpg
039.jpg
161.jpg
169.jpg
102.jpg
153.jpg
146.jpg
113.jpg
132.jpg
199.jpg
060.jpg
105.jpg
130.jpg
052.jp

In [35]:
watermark_text = "sun*"

# watermarker = DwtDctWatermark(
#     wavelet="haar",
#     block_size=8,
#     alpha=10.0
# )

watermark_bits = np.random.randint(
    0, 2,
    size=30,
    dtype=np.uint8
)

print("Watermark:", watermark_text)
print("Bits:", len(watermark_bits))

Watermark: sun*
Bits: 30


In [36]:
watermark_text_long = "Montgomery Monty McQueen, more commonly called Lightning McQueen, is a fictional anthropomorphic stock car and the protagonist of the Disney Pixar Cars franchise. He was developed by John Lasseter and co-director Joe Ranft from a story concept by Jorgen Klubien. Lightning's appearances include the feature films Cars, Cars 2, and Cars 3, as well as the animated series Cars Toons and Cars on the Road. He is also a playable character in each of the Cars video game installments. Primarily voiced by Owen Wilson, Lightning is recognizable by his red body with yellow and orange lightning bolt stickers featuring his racing number on his sides.In Cars, Lightning begins as a talented but cocky rookie in the Piston Cup racing series who becomes stranded in the small town of Radiator Springs, where he learns about humility and friendship from the locals. Over his professional racing career, he achieves several Piston Cup victories. In Cars 2, he competes in the World Grand Prix, while his friend Tow Mater is unwittingly dragged into a spy mission. In Cars 3, he struggles to come to terms with retirement and assumes the role of Cruz Ramirez's mentor.Despite receiving a mixed reaction from critics in the first film, Lightning has become a recognizable face and mascot of the Cars franchise. He has been widely merchandised in the form of branded toy cars and other products. He has been mentioned in commentary by NASCAR racing drivers, including Kyle Busch and Chris Buescher, and his achievements have been discussed by sports journalist Stephen A. Smith. Critics have described him as one of the greatest or most iconic cars in film."

watermarker = DwtDctWatermark(
    wavelet="haar",
    block_size=8,
    alpha=10.0
)

watermark_bits_long = watermarker.text_to_bits(
    watermark_text_long
)

print("Watermark:", watermark_text_long)
print("Bits:", len(watermark_bits_long))

Watermark: Montgomery Monty McQueen, more commonly called Lightning McQueen, is a fictional anthropomorphic stock car and the protagonist of the Disney Pixar Cars franchise. He was developed by John Lasseter and co-director Joe Ranft from a story concept by Jorgen Klubien. Lightning's appearances include the feature films Cars, Cars 2, and Cars 3, as well as the animated series Cars Toons and Cars on the Road. He is also a playable character in each of the Cars video game installments. Primarily voiced by Owen Wilson, Lightning is recognizable by his red body with yellow and orange lightning bolt stickers featuring his racing number on his sides.In Cars, Lightning begins as a talented but cocky rookie in the Piston Cup racing series who becomes stranded in the small town of Radiator Springs, where he learns about humility and friendship from the locals. Over his professional racing career, he achieves several Piston Cup victories. In Cars 2, he competes in the World Grand Prix, while h

In [37]:
results = []

for path in selected_paths:

    image = cv2.imread(str(path))

    if image is None:
        print(f"Could not read: {path}")
        continue

    watermarked = watermarker.embed_bits(
        image,
        watermark_bits
    )

    psnr = calculate_psnr(
        image,
        watermarked
    )

    results.append({
        "path": path,
        "original": image,
        "watermarked": watermarked,
        "psnr": psnr,
    })

    print(
        f"{path.name:30s} "
        f"PSNR = {psnr:.2f} dB"
    )

171.jpg                        PSNR = 51.26 dB
034.jpg                        PSNR = 51.17 dB
012.jpg                        PSNR = 51.04 dB
198.jpg                        PSNR = 51.08 dB
077.jpg                        PSNR = 50.99 dB
069.jpg                        PSNR = 50.87 dB
063.jpg                        PSNR = 51.12 dB
041.jpg                        PSNR = 51.02 dB
197.jpg                        PSNR = 50.83 dB
032.jpg                        PSNR = 51.14 dB
181.jpg                        PSNR = 50.99 dB
222.jpg                        PSNR = 51.14 dB
147.jpg                        PSNR = 51.10 dB
028.jpg                        PSNR = 51.17 dB
159.jpg                        PSNR = 50.94 dB
115.jpg                        PSNR = 50.64 dB
014.jpg                        PSNR = 50.98 dB
013.jpg                        PSNR = 50.99 dB
029.jpg                        PSNR = 51.13 dB
061.jpg                        PSNR = 50.80 dB
065.jpg                        PSNR = 50.85 dB
137.jpg      

In [38]:
# results_long = []

# for path in selected_paths:

#     image = cv2.imread(str(path))

#     if image is None:
#         print(f"Could not read: {path}")
#         continue

#     watermarked_long = watermarker.embed_bits(
#         image,
#         watermark_bits_long
#     )

#     psnr = calculate_psnr(
#         image,
#         watermarked_long
#     )

#     results_long.append({
#         "path": path,
#         "original": image,
#         "watermarked": watermarked_long,
#         "psnr": psnr,
#     })

#     print(
#         f"{path.name:30s} "
#         f"PSNR = {psnr:.2f} dB"
#     )

In [39]:
capacity = (
    (image.shape[0] // 2 // 8)
    * (image.shape[1] // 2 // 8)
)

capacity_long = (
    (image.shape[0] // 2 // 8)
    * (image.shape[1] // 2 // 8)
)


utilization = len(watermark_bits) / capacity

utilization_long = len(watermark_bits_long) / capacity_long

print("Watermark:", watermark_text)
print("Bits:", len(watermark_bits))
print("Capacity:", capacity)
print("Utilization:", utilization)
print()
print()
print("Watermark:", watermark_text_long)
print("Bits:", len(watermark_bits_long))
print("Capacity:", capacity_long)
print("Utilization:", utilization_long)

Watermark: sun*
Bits: 30
Capacity: 6032
Utilization: 0.004973474801061008


Watermark: Montgomery Monty McQueen, more commonly called Lightning McQueen, is a fictional anthropomorphic stock car and the protagonist of the Disney Pixar Cars franchise. He was developed by John Lasseter and co-director Joe Ranft from a story concept by Jorgen Klubien. Lightning's appearances include the feature films Cars, Cars 2, and Cars 3, as well as the animated series Cars Toons and Cars on the Road. He is also a playable character in each of the Cars video game installments. Primarily voiced by Owen Wilson, Lightning is recognizable by his red body with yellow and orange lightning bolt stickers featuring his racing number on his sides.In Cars, Lightning begins as a talented but cocky rookie in the Piston Cup racing series who becomes stranded in the small town of Radiator Springs, where he learns about humility and friendship from the locals. Over his professional racing career, he achieves several P

In [40]:
# fig, axes = plt.subplots(
#     nrows=len(results),
#     ncols=2,
#     figsize=(10, 5 * len(results))
# )

# for i, result in enumerate(results):

#     original_rgb = cv2.cvtColor(
#         result["original"],
#         cv2.COLOR_BGR2RGB
#     )

#     watermarked_rgb = cv2.cvtColor(
#         result["watermarked"],
#         cv2.COLOR_BGR2RGB
#     )

#     axes[i, 0].imshow(original_rgb)
#     axes[i, 0].set_title(
#         f"Original\n{result['path'].name}"
#     )
#     axes[i, 0].axis("off")

#     axes[i, 1].imshow(watermarked_rgb)
#     axes[i, 1].set_title(
#         f"DWT-DCT Watermarked\n"
#         f"PSNR: {result['psnr']:.2f} dB"
#     )
#     axes[i, 1].axis("off")

# plt.tight_layout()
# plt.show()

# ATTACK

In [41]:
# decoded_bits = watermarker.extract_bits(
#     result["watermarked"],
#     len(watermark_bits)
# )

# decoded_bits_long = watermarker.extract_bits(
#     result["watermarked"],
#     len(watermark_bits_long)
# )

In [42]:
def calculate_ber(original_bits, decoded_bits):
    """
    Calculate Bit Error Rate (BER).

    BER = number of incorrect bits / total number of bits
    """

    original_bits = np.asarray(original_bits).flatten()
    decoded_bits = np.asarray(decoded_bits).flatten()

    if len(original_bits) != len(decoded_bits):
        raise ValueError(
            f"Bit length mismatch: "
            f"{len(original_bits)} vs {len(decoded_bits)}"
        )

    return np.mean(original_bits != decoded_bits)

In [43]:
from noise_layers.cropout import Cropout
from noise_layers.crop import Crop
from noise_layers.dropout import Dropout
from noise_layers.resize import Resize
from noise_layers.identity import Identity

In [ ]:
import cv2
import numpy as np


def identity(x):
    return x.copy()


def crop(x, scale=(0.7, 0.9)):
    h, w = x.shape[:2]

    min_scale, max_scale = scale

    s_h = np.random.uniform(min_scale, max_scale)
    s_w = np.random.uniform(min_scale, max_scale)

    crop_h = max(1, int(h * s_h))
    crop_w = max(1, int(w * s_w))

    top = np.random.randint(
        0,
        h - crop_h + 1
    )

    left = np.random.randint(
        0,
        w - crop_w + 1
    )

    cropped = x[
        top:top + crop_h,
        left:left + crop_w
    ]

    return cv2.resize(
        cropped,
        (w, h),
        interpolation=cv2.INTER_LINEAR
    )


def cropout(x, scale):
    h, w = x.shape[:2]

    min_scale, max_scale = scale

    patch_h = max(
        1,
        int(h * np.random.uniform(min_scale, max_scale))
    )

    patch_w = max(
        1,
        int(w * np.random.uniform(min_scale, max_scale))
    )

    top = np.random.randint(
        0,
        h - patch_h + 1
    )

    left = np.random.randint(
        0,
        w - patch_w + 1
    )

    result = x.copy()

    noise = np.random.randint(
        0,
        256,
        size=(patch_h, patch_w, x.shape[2]),
        dtype=np.uint8
    )

    result[
        top:top + patch_h,
        left:left + patch_w
    ] = noise

    return result


def dropout(x, keep_ratio):
    min_keep, max_keep = keep_ratio

    keep = np.random.uniform(
        min_keep,
        max_keep
    )

    # Independent mask for every channel,
    # equivalent to torch.rand_like(x).
    mask = np.random.random(x.shape) < keep

    result = x.copy()
    result[~mask] = 0

    return result


def resize(x, scale):
    h, w = x.shape[:2]

    min_scale, max_scale = scale

    s = np.random.uniform(
        min_scale,
        max_scale
    )

    new_h = max(1, int(h * s))
    new_w = max(1, int(w * s))

    small = cv2.resize(
        x,
        (new_w, new_h),
        interpolation=cv2.INTER_LINEAR
    )

    return cv2.resize(
        small,
        (w, h),
        interpolation=cv2.INTER_LINEAR
    )

def gaussian_noise(x, mean=0.0, var=0.01):
    std = np.sqrt(var)
    if x.dtype == np.uint8:
        noise = np.random.normal(mean * 255.0, std * 255.0, x.shape)
        return np.clip(x.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    else:
        noise = np.random.normal(mean, std, x.shape).astype(x.dtype)
        return np.clip(x + noise, -1.0, 1.0)

def jpeg_compression(x, quality=50):
    """
    Applies standard JPEG compression artifacting with a specified quality factor (1-100).
    Handles uint8 [0, 255] or float [-1, 1] / [0, 1].
    """
    is_float = np.issubdtype(x.dtype, np.floating)
    
    if is_float:
        # Denormalize assuming [-1, 1] range
        img = np.clip((x + 1.0) * 127.5, 0, 255).astype(np.uint8)
    else:
        img = x

    # Encode to JPEG in memory and decode back
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)]
    _, encimg = cv2.imencode(".jpg", img, encode_param)
    decimg = cv2.imdecode(encimg, cv2.IMREAD_COLOR if img.ndim == 3 else cv2.IMREAD_UNCHANGED)

    if is_float:
        # Renormalize back to [-1, 1]
        return (decimg.astype(np.float32) / 127.5) - 1.0
    return decimg

In [ ]:
scales = [(round(s, 1), round(s, 1)) for s in [0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]]

noise_means = {
    0.1: {"mean": 0.000, "var": 0.00010},
    0.2: {"mean": 0.005, "var": 0.00063},
    0.3: {"mean": 0.010, "var": 0.00250},
    0.4: {"mean": 0.015, "var": 0.00563},
    0.5: {"mean": 0.020, "var": 0.01000},
    0.6: {"mean": 0.030, "var": 0.02250},
    0.7: {"mean": 0.040, "var": 0.04000},
    0.8: {"mean": 0.050, "var": 0.06250},
    0.9: {"mean": 0.075, "var": 0.12250},
    1.0: {"mean": 0.100, "var": 0.25000},
}

severities = [round(s, 1) for s in np.arange(0.1, 1.1, 0.1)]

jpeg_qualities = {sev: int(sev * 100) for sev in severities}  # 0.1 -> 10, ..., 1.0 -> 100

attacks = {
    # "Identity": {
    #     1.0: lambda x: identity(x)
    # },
    # "Crop": {
    #     sev: (lambda s: lambda x: crop(x, scale=(s, s)))(sev)
    #     for sev in severities
    # },
    # "Cropout": {
    #     sev: (lambda s: lambda x: cropout(x, scale=(s, s)))(sev)
    #     for sev in severities
    # },
    # "Dropout": {
    #     sev: (lambda s: lambda x: dropout(x, keep_ratio=(s, s)))(sev)
    #     for sev in severities
    # },
    # "Resize": {
    #     sev: (lambda s: lambda x: resize(x, scale=(s, s)))(sev)
    #     for sev in severities
    # },
    # "Gaussian Noise": {
    #     sev: (lambda p: lambda x: gaussian_noise(x, mean=p["mean"], var=p["var"]))(noise_means[sev])
    #     for sev in severities
    # },
    "JPEG": {
        sev: (lambda q: lambda x: jpeg_compression(x, quality=q))(jpeg_qualities[sev])
        for sev in severities
    },
}

In [49]:
attack_results = {}

for attack_name, severity_dict in attacks.items():
    print(f"\n=================== Attack: {attack_name} ===================")
    attack_results[attack_name] = {}

    for severity, attack_fn in severity_dict.items():
        print(f"\n--- Severity / Scale: {severity} ---")
        attack_results[attack_name][severity] = []

        for i, result in enumerate(results):
            watermarked = result["watermarked"]

            # Apply attack
            attacked = attack_fn(watermarked.copy())

            # Recover watermark
            decoded_bits = watermarker.extract_bits(
                attacked,
                len(watermark_bits)
            )

            # Calculate BER
            ber = calculate_ber(
                watermark_bits,
                decoded_bits
            )

            # Image preservation
            psnr = calculate_psnr(
                result["original"],
                attacked
            )

            attack_results[attack_name][severity].append({
                "image": result["path"].name,
                "attacked": attacked,
                "ber": ber,
                "psnr": psnr,
            })

            # print(
            #     f"{result['path'].name:30s} "
            #     f"BER={ber:.4f}  "
            #     f"PSNR={psnr:.2f} dB"
            # )


=================== Attack: Resize ===================

--- Severity / Scale: 0.1 ---

--- Severity / Scale: 0.2 ---

--- Severity / Scale: 0.3 ---

--- Severity / Scale: 0.4 ---

--- Severity / Scale: 0.5 ---

--- Severity / Scale: 0.6 ---

--- Severity / Scale: 0.7 ---

--- Severity / Scale: 0.8 ---

--- Severity / Scale: 0.9 ---

--- Severity / Scale: 1.0 ---

=================== Attack: Gaussian Noise ===================

--- Severity / Scale: 0.1 ---

--- Severity / Scale: 0.2 ---

--- Severity / Scale: 0.3 ---

--- Severity / Scale: 0.4 ---

--- Severity / Scale: 0.5 ---

--- Severity / Scale: 0.6 ---

--- Severity / Scale: 0.7 ---

--- Severity / Scale: 0.8 ---

--- Severity / Scale: 0.9 ---

--- Severity / Scale: 1.0 ---


In [50]:
import pandas as pd
import numpy as np

# Flatten nested attack results into a DataFrame
rows = []

for attack_name, severity_dict in attack_results.items():
    for severity, attack_data in severity_dict.items():
        # Normalize tuple severities (e.g. (0.9, 0.9) -> 0.9) for uniform numeric sorting
        sev_val = severity[0] if isinstance(severity, (tuple, list)) else severity

        for item in attack_data:
            rows.append({
                "attack": attack_name,
                "severity": sev_val,
                "image": item["image"],
                "ber": item["ber"],
                "psnr": item.get("psnr", np.nan),
                "ssim": item.get("ssim", np.nan),
            })

results_df = pd.DataFrame(rows)

# Calculate per-attack, per-severity statistics (max, mean, min)
stats_df = (
    results_df
    .groupby(["attack", "severity"], as_index=False)
    .agg(
        max_ber=("ber", "max"),
        mean_ber=("ber", "mean"),
        min_ber=("ber", "min"),
        max_psnr=("psnr", "max"),
        mean_psnr=("psnr", "mean"),
        min_psnr=("psnr", "min"),
        max_ssim=("ssim", "max"),
        mean_ssim=("ssim", "mean"),
        min_ssim=("ssim", "min"),
    )
    .sort_values(by=["attack", "severity"])
)

# Display grouped by attack
for attack_name, group in stats_df.groupby("attack"):
    print(f"\n=================== Attack: {attack_name} ===================")
    print(group.drop(columns=["attack"]).to_string(index=False))


=================== Attack: Gaussian Noise ===================
 severity  max_ber  mean_ber  min_ber  max_psnr  mean_psnr  min_psnr  max_ssim  mean_ssim  min_ssim
      0.1 0.033333  0.002151 0.000000 40.076385  39.255280 39.054722       NaN        NaN       NaN
      0.2 0.333333  0.095084 0.000000 33.722099  32.122440 32.005611       NaN        NaN       NaN
      0.3 0.466667  0.250077 0.000000 28.328957  26.261427 26.062531       NaN        NaN       NaN
      0.4 0.566667  0.329186 0.100000 25.051317  22.847731 22.565720       NaN        NaN       NaN
      0.5 0.666667  0.371582 0.133333 22.722183  20.466970 20.103878       NaN        NaN       NaN
      0.6 0.633333  0.417819 0.200000 19.430592  17.187389 16.659235       NaN        NaN       NaN
      0.7 0.666667  0.440399 0.200000 17.133820  14.948908 14.314295       NaN        NaN       NaN
      0.8 0.700000  0.448080 0.166667 15.361958  13.291205 12.660504       NaN        NaN       NaN
      0.9 0.733333  0.464516 0.20000

In [ ]:
# attack_results_long = {}

# for attack_name, attack_fn in attacks.items():

#     print(f"\n===== {attack_name} =====")

#     attack_results_long[attack_name] = []

#     for i, result in enumerate(results_long):

#         watermarked_long = result["watermarked"]

#         # Apply attack
#         attacked = attack_fn(watermarked_long)

#         # Recover watermark
#         decoded_bits_long = watermarker.extract_bits(
#             attacked,
#             len(watermark_bits_long)
#         )

#         # Calculate BER
#         ber = calculate_ber(
#             watermark_bits_long,
#             decoded_bits_long
#         )

#         # Image preservation
#         psnr = calculate_psnr(
#             result["original"],
#             attacked
#         )

#         attack_results_long[attack_name].append({
#             "image": result["path"].name,
#             "attacked": attacked,
#             "ber": ber,
#             "psnr": psnr,
#         })

#         print(
#             f"{result['path'].name:30s} "
#             f"BER={ber:.4f}  "
#             f"PSNR={psnr:.2f} dB"
#         )

In [ ]:
for attack_name, attack_data in attack_results.items():

    mean_ber = np.mean([
        x["ber"]
        for x in attack_data
    ])

    mean_psnr = np.mean([
        x["psnr"]
        for x in attack_data
    ])

    print(
        f"{attack_name:15s} | "
        f"Mean BER: {mean_ber:.4f} | "
        f"Mean PSNR: {mean_psnr:.2f} dB"
    )

Identity        | Mean BER: 0.0000 | Mean PSNR: 51.00 dB
Crop            | Mean BER: 0.5025 | Mean PSNR: 9.94 dB
Cropout         | Mean BER: 0.0296 | Mean PSNR: 9.51 dB
Dropout         | Mean BER: 0.3803 | Mean PSNR: 15.21 dB
Resize          | Mean BER: 0.0493 | Mean PSNR: 40.92 dB


In [ ]:
# for attack_name, attack_data in attack_results_long.items():

#     mean_ber = np.mean([
#         x["ber"]
#         for x in attack_data
#     ])

#     mean_psnr = np.mean([
#         x["psnr"]
#         for x in attack_data
#     ])

#     print(
#         f"{attack_name:15s} | "
#         f"Mean BER: {mean_ber:.4f} | "
#         f"Mean PSNR: {mean_psnr:.2f} dB"
#     )

In [ ]:
# from pathlib import Path
# import random
# import shutil

# SOURCE = Path("./image/images")
# OUTPUT = Path("./image_30k")

# NUM_IMAGES = 30_000
# TRAIN_RATIO = 0.8
# VAL_RATIO = 0.1
# TEST_RATIO = 0.1

# SEED = 42

# random.seed(SEED)

# extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# images = [
#     p for p in SOURCE.iterdir()
#     if p.is_file() and p.suffix.lower() in extensions
# ]

# print(f"Found {len(images)} images")

# if len(images) < NUM_IMAGES:
#     raise ValueError(
#         f"Not enough images: found {len(images)}, "
#         f"but requested {NUM_IMAGES}"
#     )

# # Randomly select 10,000 images
# selected = random.sample(images, NUM_IMAGES)

# # Shuffle the selected images before splitting
# random.shuffle(selected)

# train_end = int(NUM_IMAGES * TRAIN_RATIO)
# val_end = train_end + int(NUM_IMAGES * VAL_RATIO)

# splits = {
#     "train": selected[:train_end],
#     "val": selected[train_end:val_end],
#     "test": selected[val_end:],
# }

# for split, split_images in splits.items():
#     split_dir = OUTPUT / split
#     split_dir.mkdir(parents=True, exist_ok=True)

#     print(f"\n{split}: {len(split_images)} images")

#     for i, image in enumerate(split_images, start=1):
#         shutil.copy2(
#             image,
#             split_dir / image.name
#         )

#         if i % 1000 == 0 or i == len(split_images):
#             print(f"  Copied {i}/{len(split_images)}")

# print("\nDone!")
# print(f"Train: {len(splits['train'])}")
# print(f"Val:   {len(splits['val'])}")
# print(f"Test:  {len(splits['test'])}")
# print(f"Output: {OUTPUT.resolve()}")

Found 30000 images

train: 24000 images
  Copied 1000/24000
  Copied 2000/24000
  Copied 3000/24000
  Copied 4000/24000
  Copied 5000/24000
  Copied 6000/24000
  Copied 7000/24000
  Copied 8000/24000
  Copied 9000/24000
  Copied 10000/24000
  Copied 11000/24000
  Copied 12000/24000
  Copied 13000/24000
  Copied 14000/24000
  Copied 15000/24000
  Copied 16000/24000
  Copied 17000/24000
  Copied 18000/24000
  Copied 19000/24000
  Copied 20000/24000
  Copied 21000/24000
  Copied 22000/24000
  Copied 23000/24000
  Copied 24000/24000

val: 3000 images
  Copied 1000/3000
  Copied 2000/3000
  Copied 3000/3000

test: 3000 images
  Copied 1000/3000
  Copied 2000/3000
  Copied 3000/3000

Done!
Train: 24000
Val:   3000
Test:  3000
Output: D:\Avatar\HiDDeN\image_30k


In [ ]:
# from pathlib import Path
# import random
# import shutil

# SOURCE = Path("./cfp_flattened")
# OUTPUT = Path("./cfp_split")

# IMAGES_PER_PERSON = 10

# TRAIN_RATIO = 0.8
# VAL_RATIO = 0.1
# TEST_RATIO = 0.1

# SEED = 42

# random.seed(SEED)

# extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# images = sorted(
#     p for p in SOURCE.iterdir()
#     if p.is_file() and p.suffix.lower() in extensions
# )

# print(f"Found {len(images)} images")

# # Make groups of 10 consecutive images.
# people = []

# for i in range(0, len(images), IMAGES_PER_PERSON):
#     group = images[i:i + IMAGES_PER_PERSON]

#     # Ignore incomplete final group
#     if len(group) != IMAGES_PER_PERSON:
#         print(f"Skipping incomplete group with {len(group)} images")
#         continue

#     # Shuffle images belonging to this person
#     random.shuffle(group)

#     people.append(group)

# print(f"Found {len(people)} people")

# # Shuffle people, NOT individual images.
# random.shuffle(people)

# n_people = len(people)

# train_end = int(n_people * TRAIN_RATIO)
# val_end = train_end + int(n_people * VAL_RATIO)

# splits = {
#     "train_class": people[:train_end],
#     "val_class": people[train_end:val_end],
#     "test_class": people[val_end:],
# }

# for split, person_groups in splits.items():

#     print(f"\n{split}: {len(person_groups)} people")

#     for person_idx, person_images in enumerate(person_groups):

#         class_dir = OUTPUT / split / f"person_{person_idx:04d}"
#         class_dir.mkdir(parents=True, exist_ok=True)

#         for image in person_images:
#             shutil.copy2(
#                 image,
#                 class_dir / image.name
#             )

#         print(
#             f"  person_{person_idx:04d}: "
#             f"{len(person_images)} images"
#         )

# print("\nDone!")
# print(f"Output: {OUTPUT.resolve()}")

In [ ]:
# from pathlib import Path
# from PIL import Image

# # Mother folder containing train/, val/, test/
# DATASET_DIR = Path("./image_30k")

# SPLITS = ["train", "val", "test"]
# TARGET_SIZE = (128, 128)

# IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# total = 0

# for split in SPLITS:
#     split_dir = DATASET_DIR / split

#     if not split_dir.exists():
#         print(f"[WARNING] Missing: {split_dir}")
#         continue

#     print(f"\nProcessing {split}...")

#     for image_path in split_dir.rglob("*"):
#         if not image_path.is_file():
#             continue

#         if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
#             continue

#         try:
#             with Image.open(image_path) as img:
#                 # Convert to RGB so PNG/RGBA/etc. don't cause surprises
#                 if img.mode not in ("RGB", "L"):
#                     img = img.convert("RGB")

#                 resized = img.resize(
#                     TARGET_SIZE,
#                     Image.Resampling.LANCZOS
#                 )

#                 # Overwrite original file
#                 resized.save(image_path)

#             total += 1

#             if total % 500 == 0:
#                 print(f"Processed {total} images...")

#         except Exception as e:
#             print(f"[ERROR] {image_path}: {e}")

# print(f"\nDone. Resized {total} images to 128x128.")


Processing train...
Processed 500 images...
Processed 1000 images...
Processed 1500 images...
Processed 2000 images...
Processed 2500 images...
Processed 3000 images...
Processed 3500 images...
Processed 4000 images...
Processed 4500 images...
Processed 5000 images...
Processed 5500 images...
Processed 6000 images...
Processed 6500 images...
Processed 7000 images...
Processed 7500 images...
Processed 8000 images...
Processed 8500 images...
Processed 9000 images...
Processed 9500 images...
Processed 10000 images...
Processed 10500 images...
Processed 11000 images...
Processed 11500 images...
Processed 12000 images...
Processed 12500 images...
Processed 13000 images...
Processed 13500 images...
Processed 14000 images...
Processed 14500 images...
Processed 15000 images...
Processed 15500 images...
Processed 16000 images...
Processed 16500 images...
Processed 17000 images...
Processed 17500 images...
Processed 18000 images...
Processed 18500 images...
Processed 19000 images...
Processed 1